<div align="center">

# Search 100 million rows with an external index

Put FAISS, USearch, or hnswlib on your fse vectors, then let fse give the hits exact scores.

</div>

## When do you need this?

An exact search reads every vector, so it slows down when the store holds hundreds of millions of rows.
An approximate index (ANN, approximate nearest neighbor) finds likely hits fast, but its scores are rounded.
**fse builds no index of its own.** It prepares the vectors for the libraries you already know, then re-scores their hits exactly.
That way you need no second database process, and the scores of the hits equal fse's exact search over those hits. A row the index misses stays missed.

## The story in a table

| Step | What happens | fse call |
|---|---|---|
| 1 | Write vectors once, in shards | `ShardSink`, `encode` |
| 2 | Give the rows to an index in blocks | `iter_batches`, `VectorBatch.normalized()` |
| 3 | Ask the index for 50 likely hits | the library's own search |
| 4 | Keep 10 hits with exact scores | `VectorStore.rescore` |
| 5 | Add rows, continue, repair | `refresh`, `iter_batches(start=)`, `repair_store` |

## 1. Write a store

We use 7,200 articles from the [AG News test split](https://huggingface.co/datasets/SetFit/ag_news) and POTION-base-8M.
The first 6,000 go into the store, 1,000 arrive later, and 200 are the questions.
The same rows are saved twice: as int8 (a quarter of the size) and as float32.
Row numbers are positions in the store. They never change, because shards are only appended.
The [shared setup](README.md#run-a-recipe) installs the pinned packages. This recipe also needs `faiss-cpu`, `usearch`, and `hnswlib` from `requirements.txt`.

In [1]:
from _helpers import Recipe
import _ann
import faiss
import hnswlib
import numpy as np
from usearch.index import Index

import fse

recipe = Recipe()
work = recipe.work
encoder = recipe.encoder()
texts = recipe.news(7200)
first, later, asked = texts[:6000], texts[6000:7000], texts[7000:]
print("Vector dimensions:", encoder.output_dim)

Recipe run: 2026-11-07. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr169-normalized/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector dimensions: 256


In [2]:
def write(name, dtype, items):
    # An existing store is appended to: the sink starts at its last row.
    with fse.ShardSink(work / name, encoder, dtype=dtype, shard_rows=1500) as sink:
        encoder.encode(items, sink=sink, workers=1)


write("int8", "int8", first)
write("float32", "float32", first)
store = fse.VectorStore.open(work / "int8")
queries = encoder.encode(asked)
print("Rows:", len(store), "Questions:", len(queries))

Rows: 6000 Questions: 200


The **exact answer** is the reference for everything below.
`search(mode="exact")` computes the cosine by fse's numerical contract for every row.

In [3]:
K = 10
exact = store.search(queries, K, mode="exact")
print("Best row for the first question:", exact.ids[0, 0], "score", exact.scores[0, 0])

Best row for the first question: 3653 score 0.44228262


## 2. FAISS: normalized blocks in, exact scores out

FAISS has no cosine metric, so it gets rows of length one and an inner-product metric.
`VectorBatch.normalized()` makes such a block from a batch without decoding the store: `float32(stored) * inv_norm`, and for int8 the codes without their scale.
FAISS copies every row into its own memory, so blocks of 2,000 rows are enough.
The index (`IVF64,SQ8`) learns its 64 cells from a sample of the rows: at least 30 rows per cell, here 3,000.
For 100 million rows use a sample of 30 to 256 rows per cell, read with `take_vectors` at sorted random positions.

In [4]:
dim = encoder.output_dim
rng = np.random.default_rng(0)
sample = store.take_vectors(np.sort(rng.choice(len(store), 3000, replace=False)))
sample /= np.linalg.norm(sample, axis=1, keepdims=True)

faiss_index = faiss.index_factory(dim, "IVF64,SQ8", faiss.METRIC_INNER_PRODUCT)
faiss_index.train(sample)


def add_to_faiss(index, start=0):
    for batch in store.iter_batches(start=start, block_rows=2000):
        keys = batch.store_row + np.arange(batch.n_rows, dtype=np.int64)
        index.add_with_ids(batch.normalized(), keys)


add_to_faiss(faiss_index)
faiss_index.nprobe = 8
unit = queries / np.linalg.norm(queries, axis=1, keepdims=True)
_, faiss_ids = faiss_index.search(unit, 50)
print("Index rows:", faiss_index.ntotal, "Hits per question:", faiss_ids.shape[1])

Index rows: 6000 Hits per question: 50


Now keep 10 of the 50 hits. `rescore` takes the original questions and the hit ids, with `-1` for a missing hit.
Every question is scored against its own candidates only.

In [5]:
faiss_rescored = store.rescore(queries, faiss_ids, K)
print("Result shape:", faiss_rescored.ids.shape)

Result shape: (200, 10)


**Check the promise.** `rescore` equals an exact search over the same candidates, bit for bit.
We compare five questions, every score by its bit pattern.

In [6]:
def check_bits(candidates, result, count=5):
    for i in range(count):
        rows = sorted({int(c) for c in candidates[i] if c >= 0})
        reference = store.search(queries[i], K, mode="exact", rows=rows)
        n = len(reference.ids)  # fewer than K if the index returned fewer rows
        assert result.ids[i, :n].tobytes() == reference.ids.tobytes()
        assert _ann.bits(result.scores[i, :n]).tobytes() == _ann.bits(reference.scores).tobytes()
        assert (result.ids[i, n:] == -1).all()  # the rest of the matrix is padding


check_bits(faiss_ids, faiss_rescored)
print("rescore equals the exact search over the same candidates")

rescore equals the exact search over the same candidates


## 3. USearch: int8 codes without a copy

USearch takes int8 rows with the cosine metric directly, and `copy=False` stores only a pointer to your array.
The int8 codes of a batch work as they are: the scale cancels out of the cosine.
**Keep the batches alive as long as the index.** A freed array leaves the index with a dangling pointer, and the program can crash instead of raising an error.
Two limits: USearch also quantizes the question to int8, so its ranking differs more than by rounding, and `rescore` always uses your float32 question.
`Index.save` writes the vectors again, so a saved index holds them twice on disk.

In [7]:
held = []  # the index points into these arrays: keep them
usearch_index = Index(ndim=dim, metric="cos", dtype="i8", connectivity=16)
for batch in store.iter_batches(block_rows=2000):
    assert batch.vectors.codes.flags.c_contiguous
    held.append(batch)
    keys = (batch.store_row + np.arange(batch.n_rows)).astype(np.uint64)
    usearch_index.add(keys, batch.vectors.codes, copy=False)

matches = usearch_index.search(queries, 50)
usearch_ids = _ann.candidates_from_unsigned(matches.keys, matches.counts, len(store))
usearch_rescored = store.rescore(queries, usearch_ids, K)
check_bits(usearch_ids, usearch_rescored)
print("Index rows:", len(usearch_index), "rescore equals the exact search over its candidates")

Index rows: 6000 rescore equals the exact search over its candidates


USearch returns unsigned keys and fills missing hits with a library value.
`candidates_from_unsigned` keeps only the valid hits, checks that every key is a row of the store, and writes `-1` everywhere else.
`rescore` refuses unsigned arrays on purpose: a key above the int64 range would otherwise turn into `-1` and vanish.

## 4. hnswlib: normalized blocks, memory only

hnswlib takes float32 only and keeps the graph and every vector in memory, so it fits up to your RAM limit.
We use the inner-product space with normalized rows, so the distance is one minus the cosine.

In [8]:
hnsw = hnswlib.Index(space="ip", dim=dim)
hnsw.init_index(max_elements=len(store) + 1000, ef_construction=100, M=16)
for batch in store.iter_batches(block_rows=2000):
    hnsw.add_items(batch.normalized(), batch.store_row + np.arange(batch.n_rows))
hnsw.set_ef(100)
labels, _ = hnsw.knn_query(unit, k=50)
hnsw_ids = _ann.candidates_from_unsigned(labels, 50, len(store))
hnsw_rescored = store.rescore(queries, hnsw_ids, K)
check_bits(hnsw_ids, hnsw_rescored)
print("Index rows:", hnsw.get_current_count())

Index rows: 6000


## 5. How good are the hits?

Recall@10 is the share of the exact top 10 that the index found.
The first column ranks by the index's own scores. The second keeps its 50 candidates and ranks them exactly.
Both numbers depend on the model, the data, and the index settings. They are a quality check on this small store, not a speed result.

In [9]:
rows = [
    ("FAISS IVF64,SQ8", faiss_ids, faiss_rescored),
    ("USearch i8", usearch_ids, usearch_rescored),
    ("hnswlib", hnsw_ids, hnsw_rescored),
]
print(f"{'index':18s} {'own top 10':>10s} {'rescored':>10s}")
for name, raw, rescored in rows:
    own = _ann.recall_at_k(raw, exact.ids, K)
    after = _ann.recall_at_k(rescored.ids, exact.ids, K)
    print(f"{name:18s} {own:10.3f} {after:10.3f}")

index              own top 10   rescored
FAISS IVF64,SQ8         0.900      0.905
USearch i8              0.957      0.986
hnswlib                 0.996      0.996


Rescoring only reorders what the index found, so it can never beat the index's own recall within 50 candidates.
Its gain is the order and the scores: the 10 you keep are in fse's exact order, with fse's exact scores.

## 6. Add rows without starting over

Keep a small **sidecar file** next to every index. It names the rows the index holds and a few sample rows.
Write it only after the index is saved, and replace it atomically (the helper does). `refresh()` picks up shards that a writer published, and `iter_batches(start=)` reads only the new rows.

In [10]:
faiss_file, sidecar_path = work / "faiss.index", work / "faiss.json"
faiss.write_index(faiss_index, str(faiss_file))  # save the index first ...
_ann.write_sidecar(sidecar_path, store, len(store))  # ... then say which rows it holds
print({k: v for k, v in _ann.read_sidecar(sidecar_path).items() if k != "sample_bits"})

write("int8", "int8", later)  # a writer appends 1,000 rows
print("New rows seen by refresh:", store.refresh(), "Rows now:", len(store))

{'store': ['/var/folders/ms/lh91gwmx7sdgzc1jgl9h4d_80000gn/T/fse-recipe-3z16bt1o/int8'], 'generation': 'ae13373771b44c1a93fa0b71846890c9', 'indexed_rows': 6000, 'dtype': 'int8', 'dim': 256, 'sample_positions': [0, 5999]}
New rows seen by refresh: 1000 Rows now: 7000


## 7. Continue after an interruption

Suppose the job adds one block of new rows and saves the index, then stops before it writes the sidecar.
On restart the saved index holds rows the sidecar does not know.
Remove the keys from `indexed_rows` on, then add from there.
FAISS IVF and flat indexes support `remove_ids`; USearch has `remove`, hnswlib `mark_deleted`.
A FAISS HNSW index cannot remove, so rebuild it.

In [11]:
sidecar = _ann.read_sidecar(sidecar_path)
_ann.check_sidecar(sidecar, store)  # raises if the store is not the one we indexed
indexed = sidecar["indexed_rows"]

for batch in store.iter_batches(start=indexed, stop=indexed + 500, block_rows=500):
    keys = batch.store_row + np.arange(batch.n_rows, dtype=np.int64)
    faiss_index.add_with_ids(batch.normalized(), keys)
faiss.write_index(faiss_index, str(faiss_file))  # saved, but the sidecar still says 6000
del faiss_index  # the process dies

faiss_index = faiss.read_index(str(faiss_file))  # the restart
print("After the restart the index holds:", faiss_index.ntotal, "rows; the sidecar says", indexed)

removed = faiss_index.remove_ids(faiss.IDSelectorRange(indexed, 2**62))
add_to_faiss(faiss_index, start=indexed)
faiss.write_index(faiss_index, str(faiss_file))
_ann.write_sidecar(sidecar_path, store, len(store))
assert faiss_index.ntotal == len(store)
print("Removed", removed, "rows, continued, index rows:", faiss_index.ntotal)

After the restart the index holds: 6500 rows; the sidecar says 6000
Removed 500 rows, continued, index rows: 7000


## 8. After a repair, the index is stale

`repair_store` keeps the verified prefix of a damaged store and moves the rest aside.
If the kept prefix is shorter than the rows your index holds, the index names rows that are gone.
A later append would give those positions to other rows, so rebuild the index from the kept rows.
The sidecar and its sample bits catch this; they do not prove the rest of the content.

In [12]:
small = work / "small"
with fse.ShardSink(small, encoder, dtype="int8", shard_rows=200) as sink:
    encoder.encode(first[:600], sink=sink, workers=1)
with fse.VectorStore.open(small) as small_store:
    _ann.write_sidecar(work / "small.json", small_store, len(small_store))

damaged = sorted(small.glob("shard-*"))[-1] / "vectors.npy"
raw = bytearray(damaged.read_bytes())
raw[-1] ^= 0xFF  # one flipped bit in the last shard
damaged.write_bytes(raw)

kept = fse.repair_store(small)
print("Rows kept by repair_store:", kept, "of 600")
with fse.VectorStore.open(small) as small_store:
    try:
        _ann.check_sidecar(_ann.read_sidecar(work / "small.json"), small_store)
    except ValueError as error:
        print("Index is stale:", error)

Rows kept by repair_store: 400 of 600
Index is stale: the store holds 400 rows, the index 600: it was shortened or repaired, so rebuild the index from the kept rows


## 9. Many stores, one view

`open_view` reads several stores as one. Its positions shift when a missing part arrives later, so they make a poor key.
**Keep one index per member store** and use the row inside the member as the key.
`view_candidates` turns those keys into positions of the view, and `rescore` on the view orders everything exactly.

In [13]:
part_a, part_b = work / "part-a", work / "part-b"
for path, items in ((part_a, first[:3000]), (part_b, first[3000:])):
    with fse.ShardSink(path, encoder, dtype="int8", shard_rows=1500) as sink:
        encoder.encode(items, sink=sink, workers=1)

view = fse.VectorStore.open_view([part_a, part_b])
member_hits, member_batches = [], []
for path in (part_a, part_b):
    with fse.VectorStore.open(path) as member:
        index = Index(ndim=dim, metric="cos", dtype="i8", connectivity=16)
        for batch in member.iter_batches(block_rows=1500):
            member_batches.append(batch)  # keep the arrays alive for the index
            keys = (batch.store_row + np.arange(batch.n_rows)).astype(np.uint64)
            index.add(keys, batch.vectors.codes, copy=False)
        found = index.search(queries, 50)
        valid = np.arange(50)[None, :] < found.counts[:, None]
        member_hits.append(_ann.view_candidates(view, path, found.keys, valid))

view_candidates = np.concatenate(member_hits, axis=1)
view_result = view.rescore(queries, view_candidates, K)
view_exact = view.search(queries, K, mode="exact")
print("Recall@10 on the view:", round(_ann.recall_at_k(view_result.ids, view_exact.ids, K), 3))

Recall@10 on the view: 0.987


## Result

Three libraries read the same shards, and `rescore` gave their hits the exact scores and order of fse: the bits matched an exact search over the same candidates.
Rows could be added, a stopped job continued, and a repair was noticed.
No timing appears here on purpose. Build time and memory of your own index depend on your data and machine.

## Before using your own data

- **Memory.** HNSW needs about `d * 4 + M * 2 * 4` bytes per vector in FAISS and `4 * d` plus roughly `M * 8` to `M * 10` bytes in hnswlib ([FAISS wiki](https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index), [hnswlib parameters](https://github.com/nmslib/hnswlib/blob/v0.8.0/ALGO_PARAMS.md)). For 100 million rows and 256 dimensions that is roughly 128 GB for HNSW over float32 vectors. This is a calculation, not a measurement.
- **Training.** FAISS IVF learns from 30 to 256 rows per cell. Sample the store with `take_vectors` at sorted random positions.
- **Keys.** The position in one store is stable. Document IDs are not unique and make no key.
- **Lifetime.** With `copy=False`, hold the batches as long as the index.
- Read the [external index guide](../docs/external-indexes.md) for the details of each library.

## What next?

- [External indexes guide](../docs/external-indexes.md)
- [Vector stores in detail](../docs/architecture/store.md)
- [Document IDs](../docs/document-ids.md)

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [14]:
view.close()
store.close()
recipe.close()